# A second policy (X-VLA) and a VLM judge on the same fresh scenes

Code of **Sections V-B and V-C** (Tables V and VI). The gate does not change what a policy does, it only says go or stop, so each
policy is rolled out once per scene and joined with the saved gate decisions (the same protocol as for SmolVLA).

* **X-VLA** (`lerobot/xvla-libero`) is driven exactly as LeRobot's own evaluation drives it: LeRobot's processors, absolute
  end-effector actions, up to 800 steps.
* **VLM judge**: Qwen2.5-VL 3B (4-bit) sees the same image and instruction and answers whether the instruction can be carried out.
  Its score and threshold were fixed on the 210 development scenes before it saw the fresh scenes.
* GPU memory (8 GB): the detectors stay on the GPU; only one policy (or the VLM) is there with them (`gpu_plan`).

*The printed output below each cell is the output of our own run (paths shortened). Long progress logs are shortened to their first and last lines.*

In [ ]:
# Shared set-up (about 3 min): result folders, models, simulator and the gate's code. Run it once per session.
%run -i setup/s0_paths.py
%run -i setup/s1_models_and_simulator.py
%run -i setup/s2_engine.py
%run -i setup/s3_calibrate_camera.py
%run -i setup/s4_engine_fixes.py
%run -i setup/s5_policy_helpers.py
%run -i setup/s6_study1_setup.py
%run -i setup/s7_study2_setup.py
%run -i setup/s8_two_policies_and_judge_setup.py

## X-VLA: load (run once per session)

In [ ]:
# %% CELL v7_X1_xvla_load — the SECOND robot policy: X-VLA (lerobot/xvla-libero). Run once per session before any other X-VLA cell.
# X-VLA is driven exactly the way LeRobot's own evaluation drives it: LeRobot's processors, ABSOLUTE end-effector control,
# up to 800 steps per episode. The gate is not involved here: one rollout per scene, joined with the gate decisions afterwards.
from lerobot.policies.xvla.modeling_xvla import XVLAPolicy
from lerobot.policies.xvla.processor_xvla import make_xvla_libero_pre_post_processors
from lerobot.envs.utils import preprocess_observation
from lerobot.utils.constants import ACTION as _ACT_KEY

XVLA_ID = "lerobot/xvla-libero"
XVLA_MAX_STEPS = 800                                     # the official LeRobot X-VLA LIBERO command: --env.episode_length=800
XROLL_CSV = os.path.join(V7_DIR, "xvla_rollouts.csv")
XROLL_FIELDS = ROLL_FIELDS + ["policy", "max_steps"]

gpu_plan("xvla")                                         # SmolVLA goes to the CPU, the VLM is unloaded -> room for X-VLA
if "xpolicy" not in globals():
    xpolicy = XVLAPolicy.from_pretrained(XVLA_ID).to(device).eval()
    xpre, xpost = make_pre_post_processors(xpolicy.config, pretrained_path=XVLA_ID,
                                           preprocessor_overrides={"device_processor": {"device": device}})
    xenv_pre, xenv_post = make_xvla_libero_pre_post_processors()
_c = xpolicy.config
print("X-VLA loaded:", XVLA_ID, "| action mode", _c.action_mode, "| chunk", _c.chunk_size, "| actions used per chunk", _c.n_action_steps,
      "| denoising steps", _c.num_denoising_steps, "| dtype", _c.dtype)
print("   policy pre-steps :", [type(s).__name__ for s in xpre.steps])
print("   policy post-steps:", [type(s).__name__ for s in xpost.steps])
print("   env pre-steps    :", [type(s).__name__ for s in xenv_pre.steps], "| env post-steps:", [type(s).__name__ for s in xenv_post.steps])
gpu_report()

def _xvla_obs(env, obs):
    """raw simulator observation -> the observation LeRobot's LiberoEnv hands to a policy (same fields as LiberoEnv._format_raw_obs), batch of 1"""
    ctrl = env.env.robots[0].controller
    b = lambda x: np.asarray(x)[None]
    return {"pixels": {"image": np.ascontiguousarray(obs["agentview_image"])[None],
                       "image2": np.ascontiguousarray(obs["robot0_eye_in_hand_image"])[None]},
            "robot_state": {"eef": {"pos": b(obs["robot0_eef_pos"]), "quat": b(obs["robot0_eef_quat"]), "mat": b(ctrl.ee_ori_mat)},
                            "gripper": {"qpos": b(obs["robot0_gripper_qpos"]), "qvel": b(obs["robot0_gripper_qvel"])},
                            "joints": {"pos": b(obs["robot0_joint_pos"]), "vel": b(obs["robot0_joint_vel"])}}}

def xvla_action(env, obs, instruction):
    """one X-VLA action with the same call sequence as lerobot_eval.rollout(): env pre -> policy pre -> policy -> policy post -> env post.
    Returns 7 numbers: ABSOLUTE target position (3), absolute orientation as axis-angle (3), gripper (-1 open / +1 close)."""
    o = preprocess_observation(_xvla_obs(env, obs))
    o["task"] = [instruction]
    o = xpre(xenv_pre(o))
    with torch.inference_mode():
        a = xpolicy.select_action(o)
    a = xenv_post({_ACT_KEY: xpost(a)})[_ACT_KEY]
    a = a.to("cpu").float().numpy()[0]
    assert a.shape == (7,) and np.isfinite(a).all(), f"unexpected X-VLA action {a}"
    return a

def _xvla_reset():
    xpolicy.reset()
    for p in (xpre, xpost, xenv_pre, xenv_post):
        if hasattr(p, "reset"):
            p.reset()

def run_xvla_v7(env, task_id, state, instruction, perturb=None, max_steps=None, trace=None):
    """X-VLA rollout on exactly the scene SmolVLA gets (same setup_episode, same perturbation, same seed) + the same failure diagnostics."""
    max_steps = int(max_steps or XVLA_MAX_STEPS)
    if device != "cpu" and _on_cpu7(xpolicy):
        gpu_plan("xvla")
    obs = setup_episode(env, task_id, state, perturb=perturb, instruction=instruction)
    _xvla_reset()
    torch.manual_seed(1000 * task_id + state)
    env.env.horizon = max(int(env.env.horizon), max_steps + 200)       # the simulator refuses to step past its horizon
    tkey, gkey = _true_keys(env, obs)
    movable = [k for k in _object_keys(obs) if any(w in k for w in ("bowl", "plate", "ramekin", "cookies"))]
    z0 = {k: float(np.asarray(obs[k])[2]) for k in movable}
    lift = {k: 0.0 for k in movable}
    min_eef = float("inf")
    success, t0, step = False, time.time(), 0
    try:
        for rb in env.env.robots:
            rb.controller.use_delta = False                            # absolute targets (LeRobot's LiberoEnv does this after the settle steps too)
        for step in range(max_steps):
            act = xvla_action(env, obs, instruction)
            if trace is not None:
                trace.append({"step": step, "action": act.copy(), "eef": np.asarray(obs["robot0_eef_pos"], float).copy()})
            obs, reward, done, info = env.step(act)
            obs = apply_obs_hook(obs)
            for k in movable:
                lift[k] = max(lift[k], float(np.asarray(obs[k])[2]) - z0[k])
            if tkey:
                min_eef = min(min_eef, float(np.linalg.norm(np.asarray(obs["robot0_eef_pos"], float) - np.asarray(obs[tkey], float))))
            if done:
                success = True
                break
    finally:
        for rb in env.env.robots:
            rb.controller.use_delta = True
    t_lift = int(tkey is not None and lift.get(tkey, 0.0) > 0.04)
    other = [k[:-4] for k in movable if k != tkey and lift[k] > 0.04]
    fdist = (round(100 * float(np.linalg.norm(np.asarray(obs[tkey])[:2] - np.asarray(obs[gkey])[:2])), 1) if tkey and gkey else None)
    row = {"task_id": task_id, "state": state, "split": "fresh", "condition": perturb or "clean", "instruction": instruction,
           "success": int(success), "episode_length": step + 1, "execution_time": round(time.time() - t0, 2),
           "min_eef_to_target_cm": round(100 * min_eef, 1) if np.isfinite(min_eef) else None,
           "target_lifted": t_lift, "other_lifted": "|".join(other), "final_target_to_goal_cm": fdist,
           "policy": "xvla", "max_steps": max_steps}
    row["failure_reason"] = failure_reason(row)
    return row

def xvla_rollouts7(states, kinds, split, path=None, warn_after=20):
    """resumable X-VLA rollouts; order: condition -> state -> task (so an interrupted run is still balanced over tasks)"""
    global OBS_HOOK
    path = path or XROLL_CSV
    done = _done_keys(path, ["task_id", "state", "condition"])
    queue = [(t, s, k) for k in kinds for s in states for t in range(NUM_TASKS) if (str(t), str(s), k) not in done]
    print(f"X-VLA rollouts to run: {len(queue)} ({len(done)} already saved) -> {os.path.basename(path)}")
    t0 = time.time()
    for i, (t, s, k) in enumerate(queue):
        env, _, clean_instr = get_task_env_and_instruction(t)
        instr = perturb_instruction(clean_instr, k, s) if k in ("absent_reference", "absent_target") else clean_instr
        row = run_xvla_v7(env, t, s, instr, perturb=k if k in SCENE_PERTURBATIONS else None)
        env.close()
        OBS_HOOK = None
        row["condition"], row["split"] = k, split
        _append(path, XROLL_FIELDS, row)
        eta = (time.time() - t0) / (i + 1) * (len(queue) - i - 1) / 60
        print(f"[{i+1}/{len(queue)}] t{t} s{s} {k:18s} X-VLA success={row['success']} steps={row['episode_length']} "
              f"{row['failure_reason']} (~{eta:.0f} min left)")
        if k == "clean" and i + 1 == warn_after:
            d = pd.read_csv(path)
            d = d[d.condition == "clean"]
            if len(d) >= warn_after and d.success.mean() < 0.60:
                print(f"\n!! X-VLA succeeded on only {100 * d.success.mean():.0f}% of the first {len(d)} clean scenes. LeRobot reports about 93% on LIBERO.\n"
                      "!! The connection to X-VLA may be wrong: stop here, run the 'official check' cell and compare both outputs.\n")
    return path
print("X-VLA tools ready.")

X-VLA loaded: lerobot/xvla-libero | action mode ee6d | chunk 30 | actions used per chunk 30 | denoising steps 10 | dtype float32
   policy pre-steps : ['RenameObservationsProcessorStep', 'AddBatchDimensionProcessorStep', 'TokenizerProcessorStep', 'XVLAAddDomainIdProcessorStep', 'DeviceProcessorStep', 'NormalizerProcessorStep']
   policy post-steps: ['UnnormalizerProcessorStep', 'DeviceProcessorStep']
   env pre-steps    : ['LiberoProcessorStep', 'XVLAImageNetNormalizeProcessorStep', 'XVLAAddDomainIdProcessorStep'] | env post-steps: ['XVLARotation6DToAxisAngleProcessorStep']
   GPU memory in use: 5.1 GB (reserved 5.2 GB)
X-VLA tools ready.


## X-VLA: one clean episode (about 1 min)

In [ ]:
# %% CELL v7_X2_xvla_smoke — ONE clean episode with X-VLA (about 1 minute). Check the output before the long run.
_env, _, _instr = get_task_env_and_instruction(0)
_trace = []
_row = run_xvla_v7(_env, 0, 5, _instr, trace=_trace)
_env.close()
OBS_HOOK = None
_a0, _e0 = _trace[0]["action"], _trace[0]["eef"]
_gap = float(np.linalg.norm(_a0[:3] - _e0))
print("instruction:", _instr)
print("first action  (x y z | axis-angle | gripper):", np.round(_a0, 3))
print("gripper position at that moment            :", np.round(_e0, 3), f"| first target is {_gap:.3f} m from the gripper")
print(f"result: success={_row['success']} after {_row['episode_length']} steps ({_row['execution_time']} s) -> {_row['failure_reason']}")
if _gap > 0.5:
    print("!! The first target is far from the gripper: the absolute-action convention may not match. Do NOT start the long run; check the action convention first.")
elif _row["success"] == 1:
    print("OK: X-VLA completed the task -> you can start the long X-VLA cell.")
else:
    print("X-VLA did not finish this one episode (it can fail sometimes). The long cell warns you if the success rate is abnormally low.")

instruction: pick up the black bowl between the plate and the ramekin and place it on the plate
first action  (x y z | axis-angle | gripper): [-0.187  0.009  1.191 -2.189 -2.192  0.055 -1.   ]
gripper position at that moment            : [-1.99e-01 -1.00e-03  1.18e+00] | first target is 0.019 m from the gripper
result: success=1 after 79 steps (18.15 s) -> success
OK: X-VLA completed the task -> you can start the long X-VLA cell.


## X-VLA on the 200 valid fresh scenes (about 2 h, resumable)

In [ ]:
# %% CELL v7_X3_xvla_rollouts — X-VLA on the 200 VALID v6 test scenes (states 5-9). About 1.5-3 h. RESUMABLE (stop and re-run any time).
# Same scenes, same perturbations and same seeds as the SmolVLA rollouts of cell v6_E, so both policies can be put behind the same gates.
xvla_rollouts7(FRESH_STATES, VALID_KINDS, "fresh")
print("X-VLA VALID ROLLOUTS DONE")

X-VLA rollouts to run: 200 (0 already saved) -> xvla_rollouts.csv
[1/200] t0 s5 clean              X-VLA success=1 steps=79 success (~73 min left)
[2/200] t1 s5 clean              X-VLA success=1 steps=110 success (~81 min left)
[3/200] t2 s5 clean              X-VLA success=1 steps=100 success (~82 min left)
... (194 more lines of progress) ...
[198/200] t7 s9 occlusion          X-VLA success=1 steps=120 success (~3 min left)
[199/200] t8 s9 occlusion          X-VLA success=1 steps=95 success (~1 min left)
[200/200] t9 s9 occlusion          X-VLA success=1 steps=121 success (~0 min left)
X-VLA VALID ROLLOUTS DONE


## Both gates in front of both policies (Table VI, first rows, seconds)

In [ ]:
# %% CELL v7_X6_two_policy_results — RESULTS on the v6 test scenes (states 5-9): both gates in front of BOTH policies (seconds).
# Uses only saved files, so it can be re-run any time; rows appear as soon as their rollouts exist.
_f6 = json.load(open(FROZEN6_JSON))
print("gate: frozen v6 config", hashlib.sha1(json.dumps(_f6, sort_keys=True, default=float).encode()).hexdigest()[:10], "| frozen at", _f6["frozen_at"])
G56 = {"GDINO gate (first paper)": decide_all(load6(FRESH_GD_CSV, FRESH_STATES), gate_cfg6("gdino")),
       "hybrid gate (v6)": decide_all(load6(FRESH_HY_CSV, FRESH_STATES), gate_cfg6("hybrid"))}
P56 = {"SmolVLA": success_map7(ROLL6_CSV), "X-VLA": success_map7(XROLL_CSV)}
for _p, _m in P56.items():
    _v = [v for (t, s, k), v in _m.items() if k in VALID_KINDS and s in FRESH_STATES]
    _c = [v for (t, s, k), v in _m.items() if k == "clean" and s in FRESH_STATES]
    print(f"{_p}: {len(_v)} valid rollouts found" + (f" | success on valid scenes {pct(np.mean(_v))}% | clean scenes {pct(np.mean(_c))}% (n={len(_c)})" if _v and _c else ""))
TWO_POLICY7 = policy_gate_table7(G56, P56, "TABLE X1 - the same two gates in front of two different robot policies (v6 test scenes, states 5-9)",
                                 save_as="x_two_policy_table.csv")
per_condition7(G56, P56, "TABLE X2 - per condition").to_csv(os.path.join(V7_DIR, "x_per_condition.csv"), index=False)
print("\nscene-by-scene (success > stop > bad):")
for _p, _m in P56.items():
    if not _m:
        continue
    _first = next(iter(G56.values()))
    for _g, _d in G56.items():
        print(f"   {_p}: " + head_to_head7(_d, always_act7(_first), _m, _g, "no gate"))
    print(f"   {_p}: " + head_to_head7(G56["hybrid gate (v6)"], G56["GDINO gate (first paper)"], _m, "hybrid gate", "GDINO gate"))
# the gate's decisions do not depend on the policy: the SAME scenes are refused for both robots
_both = [k for k in P56["SmolVLA"] if k in P56["X-VLA"] and k[2] in VALID_KINDS]
if _both:
    _s, _x = np.array([P56["SmolVLA"][k] for k in _both]), np.array([P56["X-VLA"][k] for k in _both])
    print(f"\non the {len(_both)} valid scenes rolled out with both policies: SmolVLA {pct(_s.mean())}% | X-VLA {pct(_x.mean())}% | "
          f"both succeed {pct((_s & _x).mean())}% | only X-VLA {pct(((1 - _s) & _x).mean())}% | only SmolVLA {pct((_s & (1 - _x)).mean())}% "
          f"(exact McNemar p = {mcnemar_exact(int(((1 - _s) & _x).sum()), int((_s & (1 - _x)).sum())):.2e})")

gate: frozen v6 config a5f5591cec | frozen at 2026-10-02 12:17:57
SmolVLA: 200 valid rollouts found | success on valid scenes 42.5% | clean scenes 66.0% (n=50)
X-VLA: 200 valid rollouts found | success on valid scenes 62.5% | clean scenes 98.0% (n=50)

TABLE X1 - the same two gates in front of two different robot policies (v6 test scenes, states 5-9)
policy                                       SmolVLA                                                           X-VLA                                          
gate                           no gate (always acts) GDINO gate (first paper) hybrid gate (v6) no gate (always acts) GDINO gate (first paper) hybrid gate (v6)
wrong instr. acted on %                        100.0                     12.0              5.3                 100.0                     12.0              5.3
  95% CI                               (97.5, 100.0)              (7.7, 18.2)      (2.7, 10.2)         (97.5, 100.0)              (7.7, 18.2)      (2.7, 10.2)
right instr

## Optional: X-VLA without a gate on the 150 impossible instructions
*Not used in the paper. Our run was stopped after 59 of the 150 scenes.*

In [ ]:
# %% CELL v7_X4_xvla_invalid — OPTIONAL but recommended (about 1-2 h, resumable): X-VLA WITHOUT a gate on the 150 WRONG instructions.
# Shows what the stronger policy does when the instruction cannot be carried out (named object absent / relation false).
xvla_rollouts7(FRESH_STATES, INVALID_KINDS_V4, "fresh")
_x = pd.read_csv(XROLL_CSV).drop_duplicates(["task_id", "state", "condition"], keep="last")
_x = _x[_x.condition.isin(INVALID_KINDS_V4)]
_x["lifted_something"] = ((_x.target_lifted == 1) | (_x.other_lifted.fillna("") != "")).astype(int)
print("\nX-VLA without a gate, on WRONG instructions (it always moves; the question is what it does):")
print(_x.groupby("condition").agg(scenes=("success", "size"), did_the_original_task_pct=("success", lambda s: round(100 * s.mean(), 1)),
                                  lifted_an_object_pct=("lifted_something", lambda s: round(100 * s.mean(), 1)),
                                  mean_steps=("episode_length", "mean")).round(1).to_string())

X-VLA rollouts to run: 150 (200 already saved) -> xvla_rollouts.csv
[1/150] t0 s5 absent_reference   X-VLA success=1 steps=79 success (~53 min left)
[2/150] t1 s5 absent_reference   X-VLA success=1 steps=111 success (~59 min left)
[3/150] t2 s5 absent_reference   X-VLA success=1 steps=98 success (~60 min left)
... (52 more lines of progress) ...
[56/150] t5 s5 absent_target      X-VLA success=0 steps=800 wrong_object_grasped (~103 min left)
[57/150] t6 s5 absent_target      X-VLA success=1 steps=107 success (~100 min left)
[58/150] t7 s5 absent_target      X-VLA success=1 steps=124 success (~99 min left)
[59/150] t8 s5 absent_target      X-VLA success=1 steps=90 success (~96 min left)


## Optional: LeRobot's own evaluation of the X-VLA checkpoint
*Not run for the paper.*

In [ ]:
# %% CELL v7_X5_xvla_official — OPTIONAL check (about 15-25 min): LeRobot's OWN evaluation script on the same X-VLA checkpoint.
# Use it if the X-VLA success rate in our loop looks too low, or to quote "official protocol: X%" in the paper.
import subprocess, glob, shutil, sys
XOFF_DIR = os.path.join(V7_DIR, "official_xvla_eval")
XOFF_EPISODES = 2                                          # per task -> 20 episodes (start states 0 and 1)
_exe = shutil.which("lerobot-eval")
_cmd = ([_exe] if _exe else [sys.executable, "-m", "lerobot.scripts.lerobot_eval"]) + [
    f"--policy.path={XVLA_ID}", "--env.type=libero", "--env.task=libero_spatial", "--env.control_mode=absolute",
    "--env.episode_length=800", f"--eval.n_episodes={XOFF_EPISODES}", "--eval.batch_size=1", "--seed=142",
    f"--output_dir={XOFF_DIR}", "--policy.device=cuda"]
if glob.glob(os.path.join(XOFF_DIR, "**", "eval_info.json"), recursive=True):
    print("official X-VLA evaluation already done -> reading the saved result")
else:
    if os.path.exists(XOFF_DIR):
        shutil.rmtree(XOFF_DIR)                            # an unfinished earlier attempt (LeRobot refuses existing folders)
    gpu_plan("sweep")                                      # both policies leave the GPU so the script has room for its own copy
    print("running:", " ".join(_cmd))
    _p = subprocess.run(_cmd, env={**os.environ, "MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl"}, capture_output=True, text=True)
    print(_p.stdout[-2500:])
    if _p.returncode != 0:
        print("ERROR:\n", _p.stderr[-4000:])
_info = glob.glob(os.path.join(XOFF_DIR, "**", "eval_info.json"), recursive=True)
if _info:
    def _find(d, key):
        if isinstance(d, dict):
            for k, v in d.items():
                if k == key:
                    return v
                r = _find(v, key)
                if r is not None:
                    return r
        return None
    XOFF_SUCCESS = _find(json.load(open(_info[0])), "pc_success")
    _ours = pd.read_csv(XROLL_CSV) if os.path.exists(XROLL_CSV) else pd.DataFrame(columns=["condition", "success"])
    _ours = _ours[_ours.condition == "clean"]
    print(f"\nOFFICIAL LeRobot evaluation, X-VLA on LIBERO-Spatial: {XOFF_SUCCESS}% ({XOFF_EPISODES} episodes x 10 tasks, 360x360 images, states 0-{XOFF_EPISODES - 1})")
    print(f"our loop, clean scenes: {round(100 * _ours.success.mean(), 1) if len(_ours) else 'not run yet'}% ({len(_ours)} episodes, 256x256 images, states 5-9)")
    json.dump({"official_pc_success": XOFF_SUCCESS, "episodes_per_task": XOFF_EPISODES}, open(os.path.join(V7_DIR, "official_xvla_summary.json"), "w"))

## VLM: loader and the judge's questions (run once per session)

In [ ]:
# %% CELL v7_V1_vlm — the VLM (Qwen2.5-VL 3B, 4-bit): loader + the three ways it is used. Run once per session. ~1 min after download.
#   (1) JUDGE     "just ask a VLM": image + instruction -> probability that the instruction can be carried out   (the BASELINE)
#   (2) PRESENCE  "is there a <object> in this image?" -> probability                                              (helper for our gate)
#   (3) GROUNDER  "locate the <object>" -> boxes                                                                   (helper for our gate)
VLM_ID = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit"     # 2.5 GB; the 4-bit copy of Qwen/Qwen2.5-VL-3B-Instruct
VLM_SCALE = 3                                           # the 256-pixel camera image is enlarged 3x for the VLM (a VLM needs larger images)
VLM_BOX_MODE = "pixels" if "2.5" in VLM_ID else "norm1000"   # Qwen2.5-VL answers in pixels of the image it saw; Qwen2-VL / Qwen3-VL in 0-1000
if "vlm" not in globals():
    vlm, vlm_proc, _VLM_YN = None, None, None
_VLM_CACHE = {}

def vlm_load():
    """loads the VLM if it is not on the GPU (gpu_plan() unloads it whenever a robot policy needs the memory)"""
    global vlm, vlm_proc, _VLM_YN
    if vlm is None:
        from transformers import AutoProcessor, AutoModelForImageTextToText
        gpu_plan("sweep")
        if vlm_proc is None:
            vlm_proc = AutoProcessor.from_pretrained(VLM_ID)
        kw = {"device_map": {"": 0}} if device != "cpu" else {}
        if "bnb" not in VLM_ID.lower():
            kw["dtype"] = torch.bfloat16                # only for a non-quantised model (needs much more GPU memory)
        vlm = AutoModelForImageTextToText.from_pretrained(VLM_ID, **kw).eval()
        print("VLM loaded:", VLM_ID)
        gpu_report()
    if _VLM_YN is None:
        tok = vlm_proc.tokenizer
        one = lambda words: sorted({tok.encode(w, add_special_tokens=False)[0] for w in words if len(tok.encode(w, add_special_tokens=False)) == 1})
        _VLM_YN = (one(["Yes", "yes", "YES"]), one(["No", "no", "NO"]))
        assert _VLM_YN[0] and _VLM_YN[1], "could not find single tokens for Yes / No in this tokenizer"
        print("   yes tokens", _VLM_YN[0], "| no tokens", _VLM_YN[1])

def _vlm_image(image_pil):
    return image_pil.resize((image_pil.width * VLM_SCALE, image_pil.height * VLM_SCALE), Image.BICUBIC)

def _vlm_inputs(image_big, text):
    content = ([{"type": "image"}] if image_big is not None else []) + [{"type": "text", "text": text}]
    prompt = vlm_proc.apply_chat_template([{"role": "user", "content": content}], tokenize=False, add_generation_prompt=True)
    inp = vlm_proc(text=[prompt], images=[image_big], return_tensors="pt") if image_big is not None else vlm_proc(text=[prompt], return_tensors="pt")
    return inp.to(vlm.device)

def vlm_yes(image_pil, question):
    """probability that the VLM answers Yes (one forward pass; the probabilities of its first answer word). Returns (p_yes, yes+no mass)."""
    vlm_load()
    key = ("yes", hash(image_pil.tobytes()) if image_pil is not None else None, question)
    if key not in _VLM_CACHE:
        inp = _vlm_inputs(_vlm_image(image_pil) if image_pil is not None else None, question + " Answer with Yes or No.")
        with torch.inference_mode():
            try:
                logits = vlm(**inp, logits_to_keep=1).logits[0, -1]
            except TypeError:
                logits = vlm(**inp).logits[0, -1]
        p = torch.softmax(logits.float(), -1)
        py, pn = float(p[_VLM_YN[0]].sum()), float(p[_VLM_YN[1]].sum())
        if len(_VLM_CACHE) > 5000:
            _VLM_CACHE.clear()
        _VLM_CACHE[key] = (py / (py + pn + 1e-9), py + pn)
    return _VLM_CACHE[key]

def vlm_generate(image_pil, text, max_new_tokens=160):
    """free text answer (greedy). Returns (answer, (H, W) of the image as the VLM saw it, or None for text-only)."""
    vlm_load()
    inp = _vlm_inputs(_vlm_image(image_pil) if image_pil is not None else None, text)
    with torch.inference_mode():
        ids = vlm.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
    out = vlm_proc.batch_decode(ids[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    seen = None
    if image_pil is not None:
        ps = int(getattr(vlm_proc.image_processor, "patch_size", 14))
        _, gh, gw = [int(v) for v in inp["image_grid_thw"][0].tolist()]
        seen = (gh * ps, gw * ps)
    return out.strip(), seen

def vlm_present(image_pil, phrase):
    """P(the object is somewhere in the image), according to the VLM"""
    return vlm_yes(image_pil, f'Is there a {phrase.strip()} in this image?')[0]

_VLM_BOX_JSON = re.compile(r'"bbox_2d"\s*:\s*\[\s*(-?[\d.]+)\s*,\s*(-?[\d.]+)\s*,\s*(-?[\d.]+)\s*,\s*(-?[\d.]+)\s*\]')
_VLM_BOX_PAIR = re.compile(r'\(\s*(\d+)\s*,\s*(\d+)\s*\)\s*,\s*\(\s*(\d+)\s*,\s*(\d+)\s*\)')

def vlm_boxes(image_pil, phrase):
    """boxes of the object in the coordinates of image_pil (the 256-pixel image), parsed from the VLM's answer"""
    key = ("boxes", hash(image_pil.tobytes()), phrase)
    if key not in _VLM_CACHE:
        out, (H, W) = vlm_generate(image_pil, f'Locate every "{phrase.strip()}" in the image and output the bbox coordinates in JSON format. '
                                              f'If there is none, output [].')
        found = [m.groups() for m in _VLM_BOX_JSON.finditer(out)] or [m.groups() for m in _VLM_BOX_PAIR.finditer(out)]
        sx, sy = (image_pil.width / W, image_pil.height / H) if VLM_BOX_MODE == "pixels" else (image_pil.width / 1000.0, image_pil.height / 1000.0)
        boxes = []
        for x1, y1, x2, y2 in found:
            b = [float(x1) * sx, float(y1) * sy, float(x2) * sx, float(y2) * sy]
            b = [min(b[0], b[2]), min(b[1], b[3]), max(b[0], b[2]), max(b[1], b[3])]
            if b[2] - b[0] > 1 and b[3] - b[1] > 1:
                boxes.append([round(v, 1) for v in b])
        _VLM_CACHE[key] = (boxes, out)
    return _VLM_CACHE[key]

def ground_object_vlm(image_pil, phrase, box_threshold=0.10, text_threshold=None):
    """the VLM as a detector (same interface as the other detectors). A VLM gives no box score, so the confidence of every box is
    the VLM's own probability that the object is present, placed on the OWLv2 scale. Below 0.5 the object is treated as absent."""
    p = vlm_present(image_pil, phrase)
    if p < 0.5:
        return []
    conf = OWL_RAMP[0] + p * (OWL_RAMP[1] - OWL_RAMP[0])
    return [{"bbox": b, "confidence": round(float(conf), 3), "label": phrase} for b in vlm_boxes(image_pil, phrase)[0] if conf >= box_threshold]

# ---------- (1) the JUDGE baseline ----------
VLM_PROMPTS7 = {
    "direct": 'A robot is given this instruction: "{instr}". Look at the image of its workspace. Are all objects named in the instruction '
              'visible, and is the described arrangement of the objects true in the image, so that the instruction can be carried out exactly as written?',
    "short": 'Robot instruction: "{instr}". Based on the image, can this instruction be carried out exactly as stated?',
    "strict": 'A robot is given this instruction: "{instr}". The instruction is INVALID if it names an object that is not in the image, or if it '
              'describes an object position that is not true in the image. Is the instruction valid for this image?',
}
VLM_SCORES7 = list(VLM_PROMPTS7) + ["decomp"]
VLM_JUDGE_JSON = os.path.join(V7_DIR, "vlm_judge_choice.json")      # the judge's setting, fixed once on development scenes

def vlm_judge_choice7():
    return json.load(open(VLM_JUDGE_JSON)) if os.path.exists(VLM_JUDGE_JSON) else None
VLM_FIELDS7 = (["task_id", "state", "split", "kind", "valid", "instruction"] + [f"p_{k}" for k in VLM_SCORES7]
               + [f"mass_{k}" for k in VLM_PROMPTS7] + ["decomp_parts", "seconds"])

def vlm_judge_scores(image_pil, instr):
    """all judge scores for one scene. 'decomp' = product of separate questions (each object present? the relation true?)"""
    row = {}
    for name, tpl in VLM_PROMPTS7.items():
        row[f"p_{name}"], row[f"mass_{name}"] = [round(v, 4) for v in vlm_yes(image_pil, tpl.format(instr=instr))]
    parsed, goal = parse_instruction(instr), parse_goal(instr)
    parts = {}
    objs = [parsed.get("target")] + [r for r in _as_list(parsed.get("reference")) if not is_table_center(r)] + [goal.get("object")]
    for o in dict.fromkeys(o for o in objs if o):
        parts[f"present:{o}"] = vlm_present(image_pil, o)
    where = instr.lower().split(" and place")[0]
    where = re.sub(r"^(pick up|put|place|move|set)\s+(the|a|an)\s+", "", where).replace("from table center", "at the center of the table")
    parts["relation"] = vlm_yes(image_pil, f'Is there a {where} in this image?')[0]
    row["p_decomp"] = round(float(np.prod(list(parts.values()))), 4)
    row["decomp_parts"] = json.dumps({k: round(v, 3) for k, v in parts.items()})
    return row

def run_vlm_sweep7(states, tag, limit=None):
    """the VLM judge on the same scenes as the gate (it sees only the image and the instruction). Resumable -> results/v7/vlm_judge_<tag>.csv"""
    global OBS_HOOK
    path = os.path.join(V7_DIR, f"vlm_judge_{tag}.csv")
    done = _done_keys(path, ["task_id", "state", "kind"])
    queue = [(t, s, k) for t in range(NUM_TASKS) for s in states for k in KINDS if (str(t), str(s), k) not in done]
    queue = queue[:limit] if limit is not None else queue
    print(f"\n=== VLM judge [{tag}]: {len(queue)} scenes to run ({len(done)} already saved) -> {os.path.basename(path)}")
    if not queue:
        return path
    vlm_load()
    i, t_start, env, cur = 0, time.time(), None, None
    try:
        for (t, s, k) in queue:
            if t != cur:
                if env is not None:
                    env.close()
                env, _, clean_instr = get_task_env_and_instruction(t)
                cur = t
            instr = perturb_instruction(clean_instr, k, s) if k in ("absent_reference", "absent_target") else clean_instr
            obs = setup_episode(env, t, s, perturb=k if k in SCENE_PERTURBATIONS else None, instruction=instr)
            img = Image.fromarray(np.ascontiguousarray(obs["agentview_image"][::-1, ::-1]))     # exactly the image the detectors see
            t0 = time.time()
            row = {"task_id": t, "state": s, "split": tag, "kind": k, "valid": int(k not in INVALID_KINDS_V4), "instruction": instr,
                   **vlm_judge_scores(img, instr)}
            row["seconds"] = round(time.time() - t0, 2)
            _append(path, VLM_FIELDS7, row)
            i += 1
            print(f"[{i}/{len(queue)}] t{t} s{s} {k:18s} direct={row['p_direct']:.2f} short={row['p_short']:.2f} strict={row['p_strict']:.2f} "
                  f"decomp={row['p_decomp']:.2f} ({row['seconds']}s, ~{(time.time() - t_start) / i * (len(queue) - i) / 60:.1f} min left)")
        if env is not None:
            env.close()
    finally:
        OBS_HOOK = None
    return path

def vlm_decisions7(tag, states, score, thr):
    """the judge's log as a decision table with the same columns the gate tables use (executed, C = score)"""
    d = pd.read_csv(os.path.join(V7_DIR, f"vlm_judge_{tag}.csv")).drop_duplicates(["task_id", "state", "kind"], keep="last")
    d = d[d.state.isin(states)].reset_index(drop=True)
    d["C"] = d[f"p_{score}"].astype(float)
    d["executed"] = (d.C >= thr).astype(int)
    d["decision"] = np.where(d.executed == 1, "EXECUTE", "ABSTAIN")
    return d
print("VLM tools ready (the model itself loads on first use):", VLM_ID)

VLM tools ready (the model itself loads on first use): unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit


## VLM: repair of the un-quantised layers of the 4-bit checkpoint (run once per session, right after the loader)
A few layers of this checkpoint are stored un-quantised but were wrapped as 4-bit layers, which crashed. They are turned back into ordinary layers with the downloaded weights; nothing about the model's answers is tuned.

In [ ]:
# %% CELL v7_V1b_vlm_fix — run ONCE per session, right AFTER v7_V1_vlm and BEFORE v7_V2_vlm_smoke. Seconds.
# Why: a few layers of this VLM are stored un-quantised in the download, but the library wrapped them as 4-bit layers
# ("FP4 quantization state not initialized" -> AssertionError). This cell turns exactly those layers back into normal
# layers with the weights from the download. Nothing about the model's answers is changed or tuned.
import torch.nn as nn
import bitsandbytes as bnb

class _CastLinear7(nn.Module):
    """a normal linear layer that accepts whatever number format the model hands it"""
    def __init__(self, lin):
        super().__init__()
        self.lin = lin
    def forward(self, x):
        return self.lin(x.to(self.lin.weight.dtype)).to(x.dtype)

def vlm_repair():
    """replaces every 4-bit layer that has NO 4-bit data (its weights are ordinary numbers) by an ordinary layer"""
    fixed = []
    dev = next(p for p in vlm.parameters()).device
    for pname, parent in list(vlm.named_modules()):
        for cname, child in list(parent.named_children()):
            if not isinstance(child, bnb.nn.Linear4bit):
                continue
            w = child.weight
            if getattr(w, "quant_state", None) is not None or w.dim() != 2 or w.shape[1] == 1 or not w.data.is_floating_point():
                continue                                             # a real 4-bit layer -> leave it alone
            lin = nn.Linear(w.shape[1], w.shape[0], bias=child.bias is not None)
            lin.weight = nn.Parameter(w.data.detach().clone().to(dev), requires_grad=False)
            if child.bias is not None:
                lin.bias = nn.Parameter(child.bias.data.detach().clone().to(dev, lin.weight.dtype), requires_grad=False)
            setattr(parent, cname, _CastLinear7(lin).eval())
            fixed.append(f"{pname}.{cname} {tuple(w.shape)} {w.data.dtype}")
    vlm._v7_repaired = True
    return fixed

if "_vlm_load_orig7" not in globals():
    _vlm_load_orig7 = vlm_load
def vlm_load():
    """the loader of v7_V1_vlm + the repair (the VLM is reloaded whenever a robot policy has used the GPU)"""
    _vlm_load_orig7()
    if not getattr(vlm, "_v7_repaired", False):
        f = vlm_repair()
        print(f"   VLM repair: {len(f)} un-quantised layer(s) restored as normal layers")
        for s in f[:12]:
            print("     ", s)

vlm_load()
n4 = sum(isinstance(m, bnb.nn.Linear4bit) for m in vlm.modules())
bad = [n for n, m in vlm.named_modules() if isinstance(m, bnb.nn.Linear4bit) and getattr(m.weight, "quant_state", None) is None]
print(f"4-bit layers left: {n4} | still without 4-bit data: {len(bad)} {bad[:5]}")
print("OK -> now run v7_V2_vlm_smoke" if not bad else "!! some layers are still not repaired - check the list above")

   VLM repair: 2 un-quantised layer(s) restored as normal layers
      model.visual.merger.mlp.0 (5120, 5120) torch.bfloat16
      model.visual.merger.mlp.2 (2048, 5120) torch.bfloat16
4-bit layers left: 412 | still without 4-bit data: 0 []
OK -> now run v7_V2_vlm_smoke


## VLM: one scene (look at the boxes)

In [ ]:
# %% CELL v7_V2_vlm_smoke — loads the VLM and tries it on ONE scene (2-3 min including loading). Look at the picture and the text.
vlm_load()
_env, _, _instr = get_task_env_and_instruction(9)                      # "the black bowl on the wooden cabinet"
_obs = setup_episode(_env, 9, 0)
_img = Image.fromarray(np.ascontiguousarray(_obs["agentview_image"][::-1, ::-1]))
_env.close()
OBS_HOOK = None
print("1) is the object in the image?  (first four are there, last three are NOT)")
for _q in ["plate", "wooden cabinet", "stove", "cookie box", "banana", "red mug", "blue teapot"]:
    _t0 = time.time()
    print(f"   {_q:15s} P(yes) = {vlm_present(_img, _q):.2f}   ({time.time() - _t0:.1f}s)")
print("2) the judge (should be high for the first sentence, low for the other two):")
for _s in [_instr, _instr.replace("black bowl", "banana"), _instr.replace("wooden cabinet", "red mug")]:
    _r = vlm_judge_scores(_img, _s)
    print(f"   direct={_r['p_direct']:.2f} short={_r['p_short']:.2f} strict={_r['p_strict']:.2f} decomp={_r['p_decomp']:.2f} | answer mass {_r['mass_direct']:.2f} | {_s}")
print("3) boxes from the VLM (raw answer, then the picture):")
_fig, _ax = plt.subplots(1, 4, figsize=(16, 4.3))
for _a, _q in zip(_ax, ["wooden cabinet", "stove", "cookie box", "plate"]):
    _b, _raw = vlm_boxes(_img, _q)
    print(f"   {_q:15s} -> {_b}   raw: {_raw[:110]!r}")
    _a.imshow(_img); _a.set_title(_q); _a.axis("off")
    for _x1, _y1, _x2, _y2 in _b:
        _a.add_patch(patches.Rectangle((_x1, _y1), _x2 - _x1, _y2 - _y1, fill=False, edgecolor="lime", linewidth=2))
plt.tight_layout(); plt.savefig(os.path.join(V7_FIG, "vlm_smoke_boxes.png"), dpi=110); plt.show()
print("each green box must sit on the named object. If the boxes are shifted or scaled wrongly, fix this before running anything else.")
gpu_report()

## VLM judge on the 210 development scenes; its score and threshold are then FIXED (about 30 min, resumable)

In [ ]:
# %% CELL v7_V3_vlm_judge_dev — the VLM judge on the 210 development scenes (~20-35 min, resumable); then its best setting is FIXED.
print("PRE-SET RULE: of the four judge scores the one with the highest AUROC on the development scenes is used; its threshold is the one\n"
      "with the highest J on the grid 0.05 ... 0.95 (ties: closest to 0.5). The judge is tuned on development data exactly like our gate was.")
run_vlm_sweep7(DEV7_STATES, "dev")
_d = pd.read_csv(os.path.join(V7_DIR, "vlm_judge_dev.csv")).drop_duplicates(["task_id", "state", "kind"], keep="last")
_d = _d[_d.state.isin(DEV7_STATES)]
_grid = [round(0.05 * i, 2) for i in range(1, 20)]
_rows = []
for _sc in VLM_SCORES7:
    _pos, _neg = _d[_d.valid == 1][f"p_{_sc}"].astype(float), _d[_d.valid == 0][f"p_{_sc}"].astype(float)
    _j = lambda th: float((_pos >= th).mean() - (_neg >= th).mean())
    _th = max(_grid, key=lambda th: (round(_j(th), 6), -abs(th - 0.5)))
    _rows.append({"score": _sc, "AUROC": round(auroc(_pos, _neg), 3), "J at 0.5": round(100 * _j(0.5), 1), "best threshold": _th,
                  "J at best": round(100 * _j(_th), 1), "right accepted % at best": pct((_pos >= _th).mean()),
                  "wrong acted % at best": pct((_neg >= _th).mean()), "seconds per scene": round(float(_d.seconds.mean()), 2)})
VLM_DEV7 = pd.DataFrame(_rows)
print(f"\nVLM judge on {len(_d)} development scenes:")
print(VLM_DEV7.to_string(index=False))
VLM_DEV7.to_csv(os.path.join(V7_DIR, "dev_vlm_judge.csv"), index=False)
if len(_d) < len(DEV7_STATES) * NUM_TASKS * len(KINDS):
    print("\nthe development sweep is not complete yet -> run this cell again (it resumes).")
elif os.path.exists(VLM_JUDGE_JSON):
    print("\nthe judge setting is already fixed (not changed):", json.dumps(vlm_judge_choice7(), default=float)[:260])
else:
    _used = [f for f in ("vlm_judge_fresh.csv", "vlm_judge_test.csv") if os.path.exists(os.path.join(V7_DIR, f))]
    assert not _used, f"{_used} already exist - the judge setting must be fixed before it sees test scenes"
    _best = VLM_DEV7.sort_values("AUROC", ascending=False, kind="stable").iloc[0]
    json.dump({"score": _best["score"], "thr": float(_best["best threshold"]), "dev_AUROC": float(_best["AUROC"]), "dev_J": float(_best["J at best"]),
               "vlm": VLM_ID, "prompt": VLM_PROMPTS7.get(_best["score"], "product of separate questions"),
               "fixed_at": time.strftime("%Y-%m-%d %H:%M:%S")}, open(VLM_JUDGE_JSON, "w"), indent=1)
    print("\nJUDGE SETTING FIXED ->", json.dumps(vlm_judge_choice7(), default=float)[:260])
if os.path.exists(VLM_JUDGE_JSON) and len(_d):
    _c = vlm_judge_choice7()
    print("\nper condition, robot allowed to act (%), development scenes, with the fixed judge:")
    print(vlm_decisions7("dev", DEV7_STATES, _c["score"], _c["thr"]).groupby("kind").executed.mean().mul(100).round(0).reindex(KINDS).to_string())

PRE-SET RULE: of the four judge scores the one with the highest AUROC on the development scenes is used; its threshold is the one
with the highest J on the grid 0.05 ... 0.95 (ties: closest to 0.5). The judge is tuned on development data exactly like our gate was.

=== VLM judge [dev]: 210 scenes to run (0 already saved) -> vlm_judge_dev.csv
... (225 more lines of progress) ...
relation_violation    73.0
distractor            67.0
position_change       60.0
occlusion             77.0


## VLM judge on the 350 fresh scenes, next to the gates and in front of both policies (Tables V and VI)

In [ ]:
# %% CELL v7_V4_vlm_judge_fresh — the fixed VLM judge on the 350 v6 test scenes (states 5-9). ~30-45 min, resumable.
# Then: our gates vs "just ask a VLM" on the same scenes, in front of both robot policies.
_J = vlm_judge_choice7()
assert _J, "run v7_V3_vlm_judge_dev to the end first (it fixes the judge's prompt and threshold on development scenes)"
print("VLM judge setting fixed on development scenes: score =", _J["score"], "| threshold =", _J["thr"])
run_vlm_sweep7(FRESH_STATES, "fresh")
_n = len(FRESH_STATES) * NUM_TASKS * len(KINDS)
_vj = vlm_decisions7("fresh", FRESH_STATES, _J["score"], _J["thr"])
if len(_vj) < _n:
    print(f"only {len(_vj)}/{_n} scenes done -> run this cell again (it resumes)")
else:
    GA7 = {"GDINO gate (first paper)": decide_all(load6(FRESH_GD_CSV, FRESH_STATES), gate_cfg6("gdino")),
           "hybrid gate (v6)": decide_all(load6(FRESH_HY_CSV, FRESH_STATES), gate_cfg6("hybrid")),
           "VLM judge alone (Qwen2.5-VL)": _vj}
    GA7["hybrid gate AND VLM judge"] = both7(GA7["hybrid gate (v6)"], _vj)
    summary7(GA7, f"TABLE V1 - checkers on the v6 test scenes (states {FRESH_STATES})", save_as="v_judge_vs_gates.csv")
    print("\nrobot allowed to act, per condition (%):")
    print(pd.DataFrame({n: d.groupby("kind").executed.mean().mul(100).round(0) for n, d in GA7.items()}).reindex(KINDS).T.to_string())
    print("\n hybrid gate (v6)  vs  VLM judge alone")
    mcnemar7(GA7["hybrid gate (v6)"], _vj, "the hybrid gate", "the VLM judge")
    print("\nall four judge scores on these scenes (only the fixed one counts; the others are shown for completeness):")
    _all = pd.read_csv(os.path.join(V7_DIR, "vlm_judge_fresh.csv")).drop_duplicates(["task_id", "state", "kind"], keep="last")
    print(pd.DataFrame([{"score": s, "AUROC": round(auroc(_all[_all.valid == 1][f"p_{s}"], _all[_all.valid == 0][f"p_{s}"]), 3)} for s in VLM_SCORES7]).to_string(index=False))
    policy_gate_table7(GA7, {"SmolVLA": success_map7(ROLL6_CSV), "X-VLA": success_map7(os.path.join(V7_DIR, "xvla_rollouts.csv"))},
                       "TABLE V2 - every checker in front of both robot policies", save_as="v_judge_policies.csv")

VLM judge setting fixed on development scenes: score = decomp | threshold = 0.05

=== VLM judge [fresh]: 350 scenes to run (0 already saved) -> vlm_judge_fresh.csv
[1/350] t0 s5 clean              direct=0.59 short=0.56 strict=0.35 decomp=0.03 (4.15s, ~53.6 min left)
... (390 more lines of progress) ...
rolled out                                   200/200                  200/200          200/200                      200/200                   200/200               200/200                  200/200          200/200                      200/200                   200/200
all scenes: success %                           24.3                      9.1             13.4                         18.0                      10.3                  35.7                     13.4             20.0                         25.1                      14.3
all scenes: stopped %                            0.0                     70.9             67.7                         50.3                      76.9        